In [1]:
from collections import defaultdict, namedtuple
import heapq

In [2]:
sols_by_date = defaultdict(list)
with open("../data/data.csv", "r") as f:
    for line in f.readlines():
        l = line.rstrip().split(',')
        sols_by_date[l[0]].append(frozenset(l))

In [3]:
Edge = namedtuple("edge", ["src", "dest", "weight"])

In [ ]:
# Build a graph of solutions, where every solution has a path to every other solution for the next day.
# each solution is uniquely identified by a `frozenset` of the placements.

links = defaultdict(list)
start = frozenset(["start"])
prev_day = [start]
curr_day = []
for date, sols in sols_by_date.items():
    if date == '29feb': # not leap year
        continue
    for sol in sols:
        curr_day.append(sol)
        for prev in prev_day:
            links[prev].append(
                    Edge(
                    src=prev,
                    dest=sol,
                    weight=len(prev.difference(sol)) - 1
                )
            )
    prev_day = curr_day
    curr_day = []


In [11]:
# Dijkstra's algorithm to walk the graph and find the
# shortest path from 01jan to 31dec

# Takes a long time to run.

goal = "31dec"
# queue of all we need to visit, starting from node `start` for simplicity
priority_q = [(0, item, [item.dest]) for item in links[start]]
seen = {}
while priority_q:
    cost, edge, path = heapq.heappop(priority_q)
    if goal in edge.src:
        print("SOLUTION!")
        path = path + [edge.dest]
        cost = cost + edge.weight
        sol = (cost, edge, path)
        print(sol)
        print("SOLUTION!")
        break
    # If we've been here already, check if the new route is shorter
    if edge in seen and seen[edge] <= cost:
        continue
    seen[edge] = cost

    # For every node reachable from current, add it to queue
    for edge in links.get(edge.dest, ()):
        heapq.heappush(priority_q,
                        (cost + edge.weight,
                        edge,
                        path + [edge.dest]))

# free up some memory
del seen
del priority_q

In [9]:
months = ('jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec')

# drop `start` node
for i in sol[2][1:]:
    # seperate date from solution
    s = [a for a in i if a[2:] not in months]
    date = [a for a in i if a[2:] in months]
    print(f"{date[0]}, {', '.join(s)}")

01jan, 45J1, 32T2f, 51P1f, 63O0, 21X3f, 34C1, 14Z0, 15L0
02jan, 45J1, 61T3, 41O0, 21P0f, 12X2f, 34Z0f, 54C3, 15L0
03jan, 45J1, 61T3, 12X2f, 34Z0f, 54C3, 21P0, 41O0, 15L0
04jan, 45J1, 61T3, 41P0f, 12X2f, 21O0, 34Z0f, 54C3, 15L0
05jan, 45J1, 61T3, 12X2f, 34Z0f, 21O0, 41P0, 54C3, 15L0
06jan, 73P2, 52T1, 45J1, 12X2f, 34Z0f, 21O0, 41C1, 15L0
07jan, 52T1, 45J1, 12X2f, 63P2f, 21O0, 34Z0f, 41C1, 15L0
08jan, 25J1, 43C2, 12P0f, 51Z0, 32T3, 64O0, 15L0, 21X1
09jan, 12P0, 43C2, 25J1, 51Z0, 32T3, 64O0, 15L0, 21X1
10jan, 44P1f, 26T0, 51Z0, 23J1f, 12L1, 64O0, 15C0, 21X1
11jan, 34T1f, 51Z0, 54P2, 23J1f, 12L1, 64O0, 15C0, 21X1
12jan, 34T1f, 51Z0, 23J1f, 12L1, 64O0, 15C0, 21X1, 44P2f
13jan, 34T1f, 74P2, 44O0, 51Z0, 23J1f, 12L1, 15C0, 21X1
14jan, 34T1f, 44O0, 51Z0, 64P2f, 23J1f, 12L1, 15C0, 21X1
15jan, 34T3f, 44C2, 51Z0, 23J1f, 12L1, 64O0, 21X1, 25P2
16jan, 34T3f, 44C2, 51Z0, 23J1f, 12L1, 64O0, 21X1, 15P2f
17jan, 26T0, 34P1, 51Z0, 23J1f, 12L1, 64O0, 15C0, 21X1
18jan, 44C2, 51Z0, 23J1f, 12L1, 64O0, 21X1, 1